# Fleet workflow: items 3–6 — local acceptance, 6 October 2026

Grain: one actual observed machine/module/minute; Transition fit/holdout use independent whole events. Time is source-local Asia/Bangkok; pressures psi (basis not verified). Reference fitting is separate from calibration and saved historical Shadow. No confirmed healthy labels, no untouched three-day post-freeze source, no remote Server deployment. Zero Transition donor groups pass the independent event-response screen; unsafe proposals remain unavailable for transfer.

The scripts and receipts below distinguish actual SQL reads with isolated SQLite persistence, prepared historical artifact reads, manual browser actions, and fixture unit tests. Fleet Shadow selection does not activate a machine profile.

In [1]:
from pathlib import Path
import json
ROOT=Path.cwd()
assert (ROOT/'apps/api/app/fleet_stable.py').exists()
OUT=ROOT/'notebooks/outputs/fleet_workflow_20261006'
read=lambda name:json.loads((OUT/name).read_text(encoding='utf-8'))
evidence=read('real-data.json')
assert evidence['event_overlap']==0
assert evidence['steady_models_byte_values_unchanged']
assert evidence['independent_screen_passed_groups']==0
print({k:evidence[k] for k in ['groups','reference_events','heldout_events','event_overlap','independent_screen_passed_groups']})
print(evidence['holdout'])

{'groups': 144, 'reference_events': 512, 'heldout_events': 114, 'event_overlap': 0, 'independent_screen_passed_groups': 0}
{'evaluated_events': 55, 'evaluated_minutes': 2356, 'fitted_events': 512, 'healthy_review': 'PENDING', 'heldout_events': 114, 'interpretation': 'Independent historical event-response coverage, not healthy false-positive accuracy or untouched future acceptance', 'outside_minutes': 1929, 'prospective': False, 'version': 'whole-event-heldout-v1'}


In [2]:
shadow=evidence['MX058_M1_saved_shadow']
assert shadow['false_alert_rate'] is None and shadow['fault_detection_delay'] is None
assert not shadow['prospective_verified']
print({k:shadow[k] for k in ['observed_minutes','supported_minutes','flagged_minutes','flag_burden_pct','quality_minutes','unsupported_minutes','episodes_total']})
print('Known historical owner-described MX080/M6 case:',evidence['MX080_M6_owner_described_case'])

{'observed_minutes': 4320, 'supported_minutes': 3914, 'flagged_minutes': 20, 'flag_burden_pct': 0.510986203372509, 'quality_minutes': 67, 'unsupported_minutes': 339, 'episodes_total': 7}
Known historical owner-described MX080/M6 case: {'observed_minutes': 2089, 'evaluated_minutes': 2049, 'quality_minutes': 5, 'unsupported_minutes': 35, 'raw_flags': 2049, 'persistent_minutes': 2039, 'cp2_level_flags': 2049, 'steady_minutes': 873, 'steady_flags': 873}


In [3]:
from apps.api.app.fleet_bundle import load_package
final=read('final-state.json')
store=ROOT/'artifacts/fleet-stable'
old=load_package(store/'fleet-stable-v1-2f656893533033ab994e3713')
new=load_package(store/final['selected_bundle_id'])
assert old.models()==new.models()
catalog=new.priors()['transition_catalog']
fitted={e['event_id'] for g in catalog['groups'] for e in g['examples']}
validation=json.loads((store/final['selected_bundle_id']/'validation.json').read_bytes())['transition_event_holdout']
heldout={e['event_id'] for e in validation['events']}
assert not fitted&heldout
assert len(fitted)==sum(g['events'] for g in catalog['groups'])
print('Verified immutable model checksums, 43 unchanged steady profiles, unique event IDs and disjoint holdout')

Verified immutable model checksums, 43 unchanged steady profiles, unique event IDs and disjoint holdout


In [4]:
worker=read('worker-rehearsal.json')
assert not worker['server_connected'] and not worker['real_analytics_tables_written']
assert len(worker['machines'])==4
for row in worker['machines']:
    assert row['modules']==8 and row['restart_retry_pass'] and row['retry_new_source_rows']==0
    print(row)
print(read('http-load.json'))
print(read('management-qa.json'))
assert read('registry-before.json')==read('registry-after.json')
print('Actual registry unchanged; Server acceptance and confirmed healthy/future data remain open')

{'machine': '041', 'source_rows': 474, 'modules': 8, 'minutes': 64, 'retry_new_source_rows': 0, 'restart_retry_pass': True, 'events_by_type': {'CONTEXT_ASSESSMENT_CHANGED': 8, 'FLEET_OBSERVED_PHASE_CHANGED': 8, 'OBSERVATION_START': 1, 'SOURCE_GAP': 6}}
{'machine': '058', 'source_rows': 473, 'modules': 8, 'minutes': 64, 'retry_new_source_rows': 0, 'restart_retry_pass': True, 'events_by_type': {'FLEET_OBSERVED_PHASE_CHANGED': 14, 'CONTEXT_ASSESSMENT_CHANGED': 8, 'OBSERVATION_START': 1, 'SOURCE_GAP': 7, 'FLEET_PERSISTENT_EVIDENCE': 1, 'FLEET_EVIDENCE_ENDED_OR_CENSORED': 1}}
{'machine': '080', 'source_rows': 542, 'modules': 8, 'minutes': 64, 'retry_new_source_rows': 0, 'restart_retry_pass': True, 'events_by_type': {'CONTEXT_ASSESSMENT_CHANGED': 9, 'FLEET_OBSERVED_PHASE_CHANGED': 15, 'DUPLICATE_SECOND': 65, 'FLEET_ACQUISITION_REVIEW': 8, 'OBSERVATION_RESUMED': 8, 'SOURCE_GAP': 4, 'VALVE_COMMAND_CHANGED': 14, 'MODULE_STATE_CHANGED': 14, 'BUSY_CHANGED': 14, 'SV_CHANGED': 14}}
{'machine': '081

## Reproduction and source receipts

- `notebooks/support/fleet_transition_build.py`: verified prepared Parquet minutes and observed source events; reference-only fitting, separate event holdout and conservative admission screen.
- `notebooks/support/fleet_workflow_qa.py`: bounded actual MySQL SELECT slices for MX041/058/080/081, restored durable checkpoints into an isolated evidence database, repeat-delivery checks, saved Shadow assessment and ten-client HTTP test.
- `apps/api/app/fleet_training.py`: isolated process, one durable job, immutable content-addressed output. Job 410e4726133549a1b63ca70c9e0b9727 was launched manually through the web and completed in ~203 seconds.
- `deployment/FLEET_WORKFLOW_UAT_20261006.md`: manual UI evidence and remaining claims/gates.
- `deployment/SERVER_WORKER_HANDOFF_20261006.md`: next Codex setup on actual Server 10.195.17.73.

This notebook reads saved receipts and re-verifies packages; it does not fit on holdout or rerun/relabel future data as Normal.